In [22]:
from functools import partial
import numpy as np
import pandas as pd
import ipywidgets as widgets
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import datetime, timedelta
import pytz
from IPython.display import display, Markdown
import bql

# South African Swaption Volatility

Calculates the Implied (IV) and Historical (RV) volatility of ZAR Swap Rates for different tenors and tails. The Tail-Tenor, 10Y3m, means a 3 Month option on the 10 Year Swap rate, for example.

The results are presented as:
1. A heatmap showing showing IV, RV, IV-RV with percentile depending on lookback history.
2. Term structure for Tenor and Tail of IV, RV and IV-RV.
3. A line chart showing the IV and RV time series for the Tenor. <span style="color: orange;">View Cell</span> needs a selection, else the lower chart will be blank. 
4.  <span style="color: green;">Update Chart</span> must always be selected to refresh selection data.

In [23]:
bq = bql.Service()

In [24]:
# Date setup
timestamp = pd.Timestamp('now')
yesterday = pd.to_datetime(timestamp - pd.Timedelta(1, "d")).strftime('%Y-%m-%d')

# ZA Swap Rate Tickers
tickers_ZA_rates=['SASW1 BGN Curncy','SASW2 BGN Curncy','SASW3 BGN Curncy','SASW4 BGN Curncy','SASW5 BGN Curncy','SASW6 BGN Curncy','SASW7 BGN Curncy','SASW8 BGN Curncy',
                  'SASW9 BGN Curncy','SASW10 BGN Curncy']

# Swaption Volatility Ticker List
swaption_tickers = [
    'SASV0A1 ICPL Curncy', 'SASV0A2 ICPL Curncy', 'SASV0A3 ICPL Curncy', 'SASV0A4 ICPL Curncy', 'SASV0A5 ICPL Curncy', 'SASV0A6 ICPL Curncy', 'SASV0A7 ICPL Curncy', 'SASV0A8 ICPL Curncy', 'SASV0A9 ICPL Curncy', 'SASV0A10 ICPL Curncy',
    'SASV0B1 ICPL Curncy', 'SASV0B2 ICPL Curncy', 'SASV0B3 ICPL Curncy', 'SASV0B4 ICPL Curncy', 'SASV0B5 ICPL Curncy', 'SASV0B6 ICPL Curncy', 'SASV0B7 ICPL Curncy', 'SASV0B8 ICPL Curncy', 'SASV0B9 ICPL Curncy', 'SASV0B10 ICPL Curncy',
    'SASV0C1 ICPL Curncy', 'SASV0C2 ICPL Curncy', 'SASV0C3 ICPL Curncy', 'SASV0C4 ICPL Curncy', 'SASV0C5 ICPL Curncy', 'SASV0C6 ICPL Curncy', 'SASV0C7 ICPL Curncy', 'SASV0C8 ICPL Curncy', 'SASV0C9 ICPL Curncy', 'SASV0C10 ICPL Curncy',
    'SASV0F1 ICPL Curncy', 'SASV0F2 ICPL Curncy', 'SASV0F3 ICPL Curncy', 'SASV0F4 ICPL Curncy', 'SASV0F5 ICPL Curncy', 'SASV0F6 ICPL Curncy', 'SASV0F7 ICPL Curncy', 'SASV0F8 ICPL Curncy', 'SASV0F9 ICPL Curncy', 'SASV0F10 ICPL Curncy',
    'SASV0I1 ICPL Curncy', 'SASV0I2 ICPL Curncy', 'SASV0I3 ICPL Curncy', 'SASV0I4 ICPL Curncy', 'SASV0I5 ICPL Curncy', 'SASV0I6 ICPL Curncy', 'SASV0I7 ICPL Curncy', 'SASV0I8 ICPL Curncy', 'SASV0I9 ICPL Curncy', 'SASV0I10 ICPL Curncy',
    'SASV011 ICPL Curncy', 'SASV012 ICPL Curncy', 'SASV013 ICPL Curncy', 'SASV014 ICPL Curncy', 'SASV015 ICPL Curncy', 'SASV016 ICPL Curncy', 'SASV017 ICPL Curncy', 'SASV018 ICPL Curncy', 'SASV019 ICPL Curncy', 'SASV0110 ICPL Curncy',
    'SASV1F1 ICPL Curncy', 'SASV1F2 ICPL Curncy', 'SASV1F3 ICPL Curncy', 'SASV1F4 ICPL Curncy', 'SASV1F5 ICPL Curncy', 'SASV1F6 ICPL Curncy', 'SASV1F7 ICPL Curncy', 'SASV1F8 ICPL Curncy', 'SASV1F9 ICPL Curncy', 'SASV1F10 ICPL Curncy',
    'SASV021 ICPL Curncy', 'SASV022 ICPL Curncy', 'SASV023 ICPL Curncy', 'SASV024 ICPL Curncy', 'SASV025 ICPL Curncy', 'SASV026 ICPL Curncy', 'SASV027 ICPL Curncy', 'SASV028 ICPL Curncy', 'SASV029 ICPL Curncy', 'SASV0210 ICPL Curncy',
    'SASV031 ICPL Curncy', 'SASV032 ICPL Curncy', 'SASV033 ICPL Curncy', 'SASV034 ICPL Curncy', 'SASV035 ICPL Curncy', 'SASV036 ICPL Curncy', 'SASV037 ICPL Curncy', 'SASV038 ICPL Curncy', 'SASV039 ICPL Curncy', 'SASV0310 ICPL Curncy',
    'SASV041 ICPL Curncy', 'SASV042 ICPL Curncy', 'SASV043 ICPL Curncy', 'SASV044 ICPL Curncy', 'SASV045 ICPL Curncy', 'SASV046 ICPL Curncy', 'SASV047 ICPL Curncy', 'SASV048 ICPL Curncy', 'SASV049 ICPL Curncy', 'SASV0410 ICPL Curncy',
    'SASV051 ICPL Curncy', 'SASV052 ICPL Curncy', 'SASV053 ICPL Curncy', 'SASV054 ICPL Curncy', 'SASV055 ICPL Curncy', 'SASV056 ICPL Curncy', 'SASV057 ICPL Curncy', 'SASV058 ICPL Curncy', 'SASV059 ICPL Curncy', 'SASV0510 ICPL Curncy',
    'SASV071 ICPL Curncy', 'SASV072 ICPL Curncy', 'SASV073 ICPL Curncy', 'SASV074 ICPL Curncy', 'SASV075 ICPL Curncy', 'SASV076 ICPL Curncy', 'SASV077 ICPL Curncy', 'SASV078 ICPL Curncy', 'SASV079 ICPL Curncy', 'SASV0710 ICPL Curncy',
    'SASV101 ICPL Curncy', 'SASV102 ICPL Curncy', 'SASV103 ICPL Curncy', 'SASV104 ICPL Curncy', 'SASV105 ICPL Curncy', 'SASV106 ICPL Curncy', 'SASV107 ICPL Curncy', 'SASV108 ICPL Curncy', 'SASV109 ICPL Curncy', 'SASV1010 ICPL Curncy'
]

tenor_mapping = {
    'SASV0A': '1M', 'SASV0B': '2M', 'SASV0C': '3M', 'SASV0F': '6M', 'SASV0I': '9M',
    'SASV01': '1Y', 'SASV1F': '18M', 'SASV02': '2Y', 'SASV03': '3Y', 'SASV04': '4Y',
    'SASV05': '5Y', 'SASV07': '7Y', 'SASV10': '10Y'
}

tenor_windows = {
    '1M': 21, '2M': 42, '3M': 63, '6M': 126, '9M': 189, '1Y': 252,
    '18M': 378, '2Y': 504, '3Y': 756, '4Y': 1008, '5Y': 1260, '7Y': 1764, '10Y': 2520
}

ticker_to_maturity = {f'SASW{i} BGN Curncy': str(i) for i in range(1, 11)}

def create_swaption_rename_mapping():
    rename_dict = {}
    for ticker in swaption_tickers:
        base_ticker = ticker.replace(' ICPL Curncy', '')
        for tenor_pattern, tenor_name in tenor_mapping.items():
            if base_ticker.startswith(tenor_pattern):
                maturity = base_ticker[len(tenor_pattern):]
                rename_dict[ticker] = f'SA{maturity}Y_{tenor_name}_IV'
                break
    return rename_dict

In [25]:
def load_data():
    """Load and process data from BQL"""
    date_range = bq.func.range('2015-10-01', yesterday)
    px_last = {'price': bq.data.px_last(dates=date_range, fill='prev')}
    all_tickers = tickers_ZA_rates + swaption_tickers

    request = bql.Request(all_tickers, px_last)
    response = bq.execute(request)

    try:
        df_list = [x.df() for x in response if not x.df().empty]
        df = df_list[0].copy() if len(df_list) == 1 else pd.concat(df_list, ignore_index=True)
    except:
        df = bql.combined_df(response)

    if 'CURRENCY' in df.columns:
        df.drop(columns=['CURRENCY'], inplace=True)

    df = df.reset_index().pivot(index='DATE', columns='ID', values='price')
    df.reset_index(inplace=True)
    df.columns.name = None
    df['DATE'] = pd.to_datetime(df['DATE'])
    df.set_index('DATE', inplace=True)
    df = df[df.index.weekday < 5]

    # Rename swaption columns
    mapping = {k: v for k, v in create_swaption_rename_mapping().items() if k in df.columns}
    df.rename(columns=mapping, inplace=True)

    # Calculate realized volatilities
    vol_cols = {}
    for ticker in tickers_ZA_rates:
        if ticker in df.columns:
            maturity = ticker_to_maturity[ticker]
            log_ret = np.log(df[ticker] / df[ticker].shift(1))
            for tenor, window in tenor_windows.items():
                col_name = f'SA{maturity}Y_{tenor}_RV'
                vol_cols[col_name] = round(log_ret.rolling(window=window).std() * np.sqrt(252) * 100, 3)
    
    return pd.concat([df, pd.DataFrame(vol_cols, index=df.index)], axis=1)

In [26]:
# Constants
TENORS = ['1M', '2M', '3M', '6M', '9M', '1Y', '18M', '2Y', '3Y', '4Y', '5Y', '7Y', '10Y']
MATURITIES = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
TENOR_MONTHS = [1, 2, 3, 6, 9, 12, 18, 24, 36, 48, 60, 84, 120]
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def get_historical_date(lookback_months):
    if lookback_months == 0:
        return df_data.index[-1]
    target_date = df_data.index[-1] - timedelta(days=lookback_months * 30)
    available_dates = df_data.index[df_data.index <= target_date]
    return available_dates[-1] if len(available_dates) > 0 else df_data.index[0]

def calculate_percentile(series, current_value, lookback_days):
    if pd.isna(current_value):
        return np.nan
    start_date = series.index[-1] - timedelta(days=lookback_days)
    lookback_data = series[series.index >= start_date].dropna()
    return (lookback_data <= current_value).mean() * 100 if len(lookback_data) > 0 else np.nan

def calculate_spread_percentile(iv_series, rv_series, current_spread, lookback_days):
    if pd.isna(current_spread):
        return np.nan
    start_date = iv_series.index[-1] - timedelta(days=lookback_days)
    aligned = pd.DataFrame({
        'iv': iv_series[iv_series.index >= start_date],
        'rv': rv_series[rv_series.index >= start_date]
    }).dropna()
    if len(aligned) == 0:
        return np.nan
    return ((aligned['iv'] - aligned['rv']) <= current_spread).mean() * 100

In [27]:
# Create the time series figure with empty traces - will be updated via batch_update
# Top: IV (blue) and RV (red), Bottom: Spread (green)

time_series_fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Volatility: Select a cell', 'IV-RV Spread'),
    vertical_spacing=0.12,
    row_heights=[0.6, 0.4]
)

# Trace 0: IV line
time_series_fig.add_trace(
    go.Scatter(
        name='IV',
        line=dict(color='blue', width=2),
        hovertemplate='%{x}<br>IV: %{y:.1f}%<extra></extra>'
    ), row=1, col=1
)

# Trace 1: RV line
time_series_fig.add_trace(
    go.Scatter(
        name='RV',
        line=dict(color='red', width=2),
        hovertemplate='%{x}<br>RV: %{y:.1f}%<extra></extra>'
    ), row=1, col=1
)

# Trace 2: Spread line
time_series_fig.add_trace(
    go.Scatter(
        name='Spread',
        line=dict(color='green', width=2),
        hovertemplate='%{x}<br>Spread: %{y:.1f}%<extra></extra>'
    ), row=2, col=1
)

time_series_fig.update_layout(
    height=500,
    width=1000,
    template='plotly_dark',
    margin=dict(l=60, r=60, t=40, b=40),
    hovermode='x unified',
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
time_series_fig.update_yaxes(title_text="Volatility (%)", row=1, col=1)
time_series_fig.update_yaxes(title_text="Spread (%)", row=2, col=1)
time_series_fig.update_xaxes(title_text="Date", row=2, col=1)

# Convert to FigureWidget
time_series_fig = go.FigureWidget(time_series_fig)

In [34]:
def create_volatility_heatmap(lookback_months):
    """Create volatility heatmap - EXACT logic from working code"""
    lookback_days = lookback_months * 30
    
    current_iv = np.full((len(TENORS), len(MATURITIES)), np.nan)
    hover_text = np.full((len(TENORS), len(MATURITIES)), "", dtype=object)
    annotations = []
    
    for i, tenor in enumerate(TENORS):
        for j, maturity in enumerate(MATURITIES):
            iv_col, rv_col = f'SA{maturity}Y_{tenor}_IV', f'SA{maturity}Y_{tenor}_RV'
            if iv_col in df_data.columns and rv_col in df_data.columns:
                iv_s, rv_s = df_data[iv_col].dropna(), df_data[rv_col].dropna()
                if len(iv_s) > 0 and len(rv_s) > 0:
                    curr_iv, curr_rv = iv_s.iloc[-1], rv_s.iloc[-1]
                    perc_iv = calculate_percentile(iv_s, curr_iv, lookback_days)
                    perc_rv = calculate_percentile(rv_s, curr_rv, lookback_days)
                    current_iv[i, j] = curr_iv
                    hover_text[i, j] = f"Tenor: {tenor}, Maturity: {maturity}Y<br>IV: {curr_iv:.1f}% (Pctl: {perc_iv:.1f})<br>RV: {curr_rv:.1f}% (Pctl: {perc_rv:.1f})"
                    annotations.append(dict(x=j, y=i, text=f"<b>{curr_iv:.1f}</b> <b>{curr_rv:.1f}</b><br>{perc_iv:.1f} {perc_rv:.1f}", showarrow=False, font=dict(size=9, color='black')))
    
    fig = go.FigureWidget(data=go.Heatmap(z=current_iv, x=[f"{m}Y" for m in MATURITIES], y=TENORS,
        colorscale='RdYlBu_r', text=hover_text, hovertemplate='%{text}<extra></extra>',
        colorbar=dict(title=dict(text="Current IV (%)", side="right"))))
    fig.update_layout(
        annotations=annotations,
        title=f"ZAR Swaption Volatility Grid - {lookback_months}M Lookback",
        xaxis=dict(title="Underlying Swap Maturity", side="bottom"),
        yaxis=dict(title="Option Tenor"),
        width=1000, height=650, template='plotly_dark',
        margin=dict(l=60, r=100, t=50, b=50)
    )
    fig.add_annotation(
        text="<b>IV</b>  <b>RV</b><br>IV Pctl  RV Pctl",
        xref="paper", yref="paper", x=1.12, y=1.05,
        showarrow=False, font=dict(size=9, color='white'), align="left"
    )
    return fig

def create_spread_heatmap(lookback_months):
    """Create spread heatmap - EXACT logic from working code"""
    lookback_days = lookback_months * 30
    
    spread_matrix = np.full((len(TENORS), len(MATURITIES)), np.nan)
    hover_text = np.full((len(TENORS), len(MATURITIES)), "", dtype=object)
    annotations = []
    
    for i, tenor in enumerate(TENORS):
        for j, maturity in enumerate(MATURITIES):
            iv_col, rv_col = f'SA{maturity}Y_{tenor}_IV', f'SA{maturity}Y_{tenor}_RV'
            if iv_col in df_data.columns and rv_col in df_data.columns:
                iv_s, rv_s = df_data[iv_col].dropna(), df_data[rv_col].dropna()
                if len(iv_s) > 0 and len(rv_s) > 0:
                    curr_spread = iv_s.iloc[-1] - rv_s.iloc[-1]
                    perc = calculate_spread_percentile(iv_s, rv_s, curr_spread, lookback_days)
                    spread_matrix[i, j] = curr_spread
                    hover_text[i, j] = f"Tenor: {tenor}, Mat: {maturity}Y<br>Spread: {curr_spread:.1f}% (Pctl: {perc:.1f})"
                    annotations.append(dict(x=j, y=i, text=f"<b>{curr_spread:.1f}</b><br>{perc:.1f}", showarrow=False, font=dict(size=10, color='black')))
    
    fig = go.FigureWidget(data=go.Heatmap(z=spread_matrix, x=[f"{m}Y" for m in MATURITIES], y=TENORS,
        colorscale='RdBu_r', text=hover_text, hovertemplate='%{text}<extra></extra>',
        colorbar=dict(title=dict(text="IV-RV Spread (%)", side="right")), zmid=0))
    fig.update_layout(
        annotations=annotations,
        title=f"ZAR Swaption IV-RV Spread Grid - {lookback_months}M Lookback",
        xaxis=dict(title="Underlying Swap Maturity", side="bottom"),
        yaxis=dict(title="Option Tenor"),
        width=1000, height=650, template='plotly_dark',
        margin=dict(l=60, r=100, t=50, b=50)
    )
    fig.add_annotation(
        text="<b>IV-RV</b><br>Spread Pctl",
        xref="paper", yref="paper", x=1.12, y=1.05,
        showarrow=False, font=dict(size=9, color='white'), align="left"
    )
    return fig

def create_term_structure_plot(vol_types, lookback_months):
    """Create IV or RV term structure plot - EXACT logic from working code"""
    target_date = get_historical_date(lookback_months)
    fig = go.FigureWidget()
    
    for vol_type in vol_types:
        for i, maturity in enumerate(MATURITIES):
            vols, valid_months, valid_tenors = [], [], []
            for j, tenor in enumerate(TENORS):
                col = f'SA{maturity}Y_{tenor}_{vol_type}'
                if col in df_data.columns:
                    data = df_data[col].loc[:target_date].dropna()
                    if len(data) > 0:
                        vols.append(data.iloc[-1])
                        valid_months.append(TENOR_MONTHS[j])
                        valid_tenors.append(tenor)
            if vols:
                line_name = f'{maturity}Y {vol_type}' if len(vol_types) > 1 else f'{maturity}Y'
                line_style = 'solid' if vol_type == 'IV' else 'dash'
                fig.add_trace(go.Scatter(x=valid_months, y=vols, mode='lines+markers', name=line_name,
                    line=dict(color=COLORS[i % len(COLORS)], dash=line_style, width=2), marker=dict(size=6),
                    hovertemplate=f'<b>{line_name}</b><br>Tenor: %{{text}}<br>{vol_type}: %{{y:.1f}}%<extra></extra>', text=valid_tenors))
    
    title_vol = "Implied Volatility" if vol_types == ['IV'] else "Realized Volatility"
    time_desc = "Current" if lookback_months == 0 else f"{lookback_months} Months Ago"
    fig.update_layout(
        title=f'Swaption {title_vol} Term Structure - {time_desc} ({target_date.strftime("%Y-%m-%d")})',
        xaxis=dict(title='Option Tenor (Months)', type='log', tickmode='array',
            tickvals=[1, 2, 3, 6, 9, 12, 18, 24, 36, 48, 60, 84, 120],
            ticktext=['1M', '2M', '3M', '6M', '9M', '1Y', '18M', '2Y', '3Y', '4Y', '5Y', '7Y', '10Y']),
        yaxis=dict(title='Volatility (%)'),
        width=1000, height=600, hovermode='closest', template='plotly_dark',
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=60, r=100, t=50, b=50)
    )
    return fig

def create_spread_term_structure(lookback_months):
    """Create IV-RV spread term structure plot - EXACT logic from working code"""
    target_date = get_historical_date(lookback_months)
    fig = go.FigureWidget()
    
    for i, maturity in enumerate(MATURITIES):
        spreads, valid_months, valid_tenors = [], [], []
        for j, tenor in enumerate(TENORS):
            iv_col = f'SA{maturity}Y_{tenor}_IV'
            rv_col = f'SA{maturity}Y_{tenor}_RV'
            if iv_col in df_data.columns and rv_col in df_data.columns:
                iv_data = df_data[iv_col].loc[:target_date].dropna()
                rv_data = df_data[rv_col].loc[:target_date].dropna()
                if len(iv_data) > 0 and len(rv_data) > 0:
                    spread = iv_data.iloc[-1] - rv_data.iloc[-1]
                    spreads.append(spread)
                    valid_months.append(TENOR_MONTHS[j])
                    valid_tenors.append(tenor)
        if spreads:
            fig.add_trace(go.Scatter(x=valid_months, y=spreads, mode='lines+markers', name=f'{maturity}Y',
                line=dict(color=COLORS[i % len(COLORS)], width=2), marker=dict(size=6),
                hovertemplate=f'<b>{maturity}Y</b><br>Tenor: %{{text}}<br>IV-RV: %{{y:.1f}}%<extra></extra>', text=valid_tenors))
    
    time_desc = "Current" if lookback_months == 0 else f"{lookback_months} Months Ago"
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.7)
    fig.update_layout(
        title=f'Swaption IV-RV Spread Term Structure - {time_desc} ({target_date.strftime("%Y-%m-%d")})',
        xaxis=dict(title='Option Tenor (Months)', type='log', tickmode='array',
            tickvals=[1, 2, 3, 6, 9, 12, 18, 24, 36, 48, 60, 84, 120],
            ticktext=['1M', '2M', '3M', '6M', '9M', '1Y', '18M', '2Y', '3Y', '4Y', '5Y', '7Y', '10Y']),
        yaxis=dict(title='IV-RV Spread (%)'),
        width=1000, height=600, hovermode='closest', template='plotly_dark',
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=60, r=100, t=50, b=50)
    )
    return fig

In [35]:
def update_time_series(tenor, maturity):
    """Update time series figure using batch_update - like commodity update_subplots"""
    iv_col = f'SA{maturity}Y_{tenor}_IV'
    rv_col = f'SA{maturity}Y_{tenor}_RV'
    
    with time_series_fig.batch_update():
        if iv_col in df_data.columns:
            iv_data = df_data[iv_col].dropna()
            time_series_fig.data[0].x = iv_data.index
            time_series_fig.data[0].y = iv_data.values
            time_series_fig.data[0].name = f'{maturity}Y {tenor} IV'
        
        if rv_col in df_data.columns:
            rv_data = df_data[rv_col].dropna()
            time_series_fig.data[1].x = rv_data.index
            time_series_fig.data[1].y = rv_data.values
            time_series_fig.data[1].name = f'{maturity}Y {tenor} RV'
        
        if iv_col in df_data.columns and rv_col in df_data.columns:
            aligned = pd.DataFrame({'iv': df_data[iv_col], 'rv': df_data[rv_col]}).dropna()
            spread = aligned['iv'] - aligned['rv']
            time_series_fig.data[2].x = spread.index
            time_series_fig.data[2].y = spread.values
            time_series_fig.data[2].name = f'{maturity}Y {tenor} Spread'
        
        # Update subplot title
        time_series_fig.layout.annotations[0].update(text=f'Volatility: {maturity}Y Swap, {tenor} Option')

In [36]:
# Create widgets
chart_picker = widgets.Dropdown(
    description='Chart Type',
    options=[
        ('Volatility Heatmap', 'heatmap_volatility'),
        ('IV-RV Spread Heatmap', 'heatmap_spread'),
        ('IV Term Structure', 'term_structure_iv'),
        ('RV Term Structure', 'term_structure_rv'),
        ('IV-RV Spread Term Structure', 'term_structure_spread')
    ],
    value='heatmap_volatility',
    layout={'width': '280px'}
)

lookback_picker = widgets.Dropdown(
    description='Lookback',
    options=[('Current', 0), ('3 Months', 3), ('6 Months', 6), ('12 Months', 12), ('24 Months', 24), ('5 Years', 60)],
    value=3,
    layout={'width': '200px'}
)

cell_selector = widgets.Dropdown(
    description='View Cell',
    options=[('Select cell...', None)] + [
        (f'{mat}Y {tenor}', f'{mat}Y_{tenor}')
        for mat in MATURITIES for tenor in TENORS
    ],
    value=None,
    layout={'width': '200px'}
)

refresh_button = widgets.Button(description='Update Chart', button_style='success', layout={'width': '120px'})
clear_button = widgets.Button(description='Clear History', button_style='warning', layout={'width': '120px'})

spinner = widgets.HTML(
    '<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>',
    layout={'visibility': 'hidden', 'margin': '0 0 0 10px'}
)

exception_box = widgets.HBox()
fig_box = widgets.VBox()

# Store current main figure reference
current_main_fig = [None]

def run(event=None):
    """Main run function - matches commodity pattern"""
    spinner.layout.visibility = 'visible'
    exception_box.children = []
    
    try:
        chart_type = chart_picker.value
        lookback = lookback_picker.value
        
        if chart_type == 'heatmap_volatility':
            main_fig = create_volatility_heatmap(lookback)
        elif chart_type == 'heatmap_spread':
            main_fig = create_spread_heatmap(lookback)
        elif chart_type == 'term_structure_iv':
            main_fig = create_term_structure_plot(['IV'], lookback)
        elif chart_type == 'term_structure_rv':
            main_fig = create_term_structure_plot(['RV'], lookback)
        elif chart_type == 'term_structure_spread':
            main_fig = create_spread_term_structure(lookback)
        
        current_main_fig[0] = main_fig
        fig_box.children = [main_fig, time_series_fig]
    
    except Exception as e:
        exception_box.children = [widgets.Label(f'{e}')]
    
    finally:
        spinner.layout.visibility = 'hidden'

def on_cell_change(change):
    if change['new'] is not None:
        mat, tenor = change['new'].split('_')
        mat = mat.replace('Y', '')
        update_time_series(tenor, mat)

def on_clear(b):
    with time_series_fig.batch_update():
        for i in range(3):
            time_series_fig.data[i].x = []
            time_series_fig.data[i].y = []
        time_series_fig.layout.annotations[0].update(text='Volatility: Select a cell')
    cell_selector.value = None

refresh_button.on_click(run)
clear_button.on_click(on_clear)
cell_selector.observe(on_cell_change, names='value')

In [37]:
# Load data
#print('Loading data...')
df_data = load_data()
#print(f'Loaded {len(df_data)} rows')

In [38]:
# Assemble the UI display - matching commodity pattern
ui_display = widgets.VBox([
    widgets.VBox([
        chart_picker,
        lookback_picker,
        cell_selector
    ]),
    widgets.HBox([refresh_button, clear_button, spinner, exception_box]),
    fig_box,
])

# Run on startup
run()
ui_display

In [39]:
#import nbconvert

# Convert notebook to Python script
#!jupyter nbconvert --to script ZAR_Swaption_GRID.ipynb

[NbConvertApp] Converting notebook ZAR_Swaption_GRID.ipynb to script
[NbConvertApp] Writing 23645 bytes to ZAR_Swaption_GRID.py
